# Praktikum 3
HNSW (Hierarchical Navigable Small World) dengan `hnswlib`

Install `hnswlib` terlebih dahulu. Percobaan berikut akan membandingkan exact NN dengan HNSW pada 1000 data 2D.

In [1]:
# !pip install -q hnswlib

In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)    # Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

=== Exact NN ===
Indices: [[447 952 896 153 757]]
Distances: [[0.02706967 0.03510858 0.03573728 0.03906028 0.04848126]]
Waktu: 0.03518104553222656 detik

=== HNSW ===
Indices: [[447 952 896 153 757]]
Distances: [[0.00073277 0.00123261 0.00127715 0.00152571 0.00235043]]
Waktu: 0.00010514259338378906 detik


### Percobaan Tambahan
Lakukan percobaan pada **metric distance** yang berbeda, **1000 vs 1 juta data**, dan **2D vs 5D data**. Catat hasilnya pada tabel.

In [3]:
import pandas as pd

def bench_hnsw(data, query, k, space, ef):
    n = data.shape[0]
    p = hnswlib.Index(space=space, dim=data.shape[1])
    p.init_index(max_elements=n, ef_construction=200, M=16)
    p.add_items(data)
    p.set_ef(ef)
    start = time.time()
    labels, dist = p.knn_query(query, k=k)
    t = (time.time() - start) * 1000
    return labels, t

rows = []
for space in ['l2', 'cosine']:
    for n_data in [1000, 1000000]:
        for d in [2, 5]:
            np.random.seed(42)
            Xd = np.random.random((n_data, d)).astype(np.float32)
            qd = np.random.random((1, d)).astype(np.float32)

            metric = 'euclidean' if space == 'l2' else 'cosine'
            nn = NearestNeighbors(n_neighbors=5, algorithm='brute', metric=metric).fit(Xd)
            start = time.time()
            _, e_idx = nn.kneighbors(qd)
            t_exact = (time.time() - start) * 1000

            h_idx, t_hnsw = bench_hnsw(Xd, qd, 5, space, ef=50)

            rows.append({
                'Distance Metrics': space,
                'Jumlah data': n_data,
                'Dimensi': d,
                'Exact NN idx': list(map(int, e_idx[0])),
                'HNSW idx': list(map(int, h_idx[0])),
                'Waktu Exact (ms)': round(t_exact, 4),
                'Waktu HNSW (ms)': round(t_hnsw, 4),
            })

tabel_hnsw = pd.DataFrame(rows)
tabel_hnsw

,Distance Metrics,Jumlah data,Dimensi,Exact NN idx,HNSW idx,Waktu Exact (ms),Waktu HNSW (ms)
0,l2,1000,2,"[112, 535, 777, 246, 763]","[112, 535, 777, 246, 763]",3.1209,0.0460
1,l2,1000,5,"[988, 780, 27, 943, 93]","[988, 780, 27, 943, 93]",0.8361,0.0477
2,l2,1000000,2,"[132774, 119034, 511191, 482199, 913140]","[132774, 119034, 511191, 482199, 913140]",13.0959,0.0629
3,l2,1000000,5,"[901095, 495168, 561017, 163986, 185734]","[901095, 495168, 561017, 163986, 185734]",17.1399,0.1030
4,cosine,1000,2,"[738, 183, 561, 91, 244]","[738, 183, 561, 91, 244]",2.0242,0.0348
5,cosine,1000,5,"[988, 943, 780, 332, 27]","[988, 943, 780, 332, 27]",1.6401,0.0551
6,cosine,1000000,2,"[446148, 298349, 124574, 298768, 445939]","[558965, 652709, 714705, 785671, 944275]",55.2588,0.0839
7,cosine,1000000,5,"[653774, 135588, 561017, 901095, 211473]","[653774, 135588, 561017, 901095, 211473]",64.8301,0.0839


> **Catatan:** `ef` (set_ef) mengontrol trade-off antara kecepatan dan akurasi. Semakin besar `ef`, semakin akurat (mendekati exact) tetapi semakin lambat.